# Module 2: Multi-Cohort Conversion Funnel, Stochastic Markov Drop Hazard & Checkout Retries
**SQL Reference:** [`sql/02_funnel_and_markov.sql`](../sql/02_funnel_and_markov.sql) | **Visual Chart:** [`figures/funnel_comparison.png`](../figures/funnel_comparison.png)

## Executive Summary (`H1` & `H2` Confirmed)
1. **Two-Proportion Z-Test (`H1`):** iOS converts at **`12.48%`** (`16,088 / 128,872`) vs. Compatible Android at **`9.65%`** (`8,258 / 85,603`), proving a highly significant **`+2.84 pp` unrounded (`+2.83 pp` rounded, `+29.4%` relative)** iOS superiority ($Z = 20.28, p < 0.0001$).
2. **Pinpointing the Exact Funnel Bottlenecks:**
   - **Compatible Android Outperforms at Initial Engagement (`S0 -> S1`):** `52.18%` of Compatible Android sessions select a plan (`click_to_order_1`) vs. `48.52%` on iOS (`+3.66 pp` Android lead).
   - **Dual Bottleneck on Compatible Android (`S1 -> S2` and `S2 -> S3`):** Compatible Android suffers a **`43.42%` drop hazard** during Plan Configuration (`S1 -> S2`, vs. `30.58%` on iOS; `+12.84 pp` excess loss) and a **`67.32%` drop hazard** at Payment (`S2 -> S3`, vs. `62.94%` on iOS; `+4.38 pp` excess loss).
3. **Incompatible Android Terminal Checkout Wall (`H2`):** `7,889` incompatible sessions select a plan (`S1`) and **`4,735` reach `click_to_order_2` (`S2`)**, where **`97.68%` (`4,625 / 4,735`) abandon** and **`110` sessions (`111` orders, `$1,238.19`) still pay** for an unusable eSIM.
4. **Upper-Bound Hardware Opportunity Sizing:** If those `27,073` incompatible sessions converted at the `9.6469%` Compatible Android benchmark (`2,612` buyers vs. `110` actual = `2,502` net incremental buyers at `$14.1615` ARPPU), Saily would capture **`+$35,751.65`** over 27 days (**`~$483,300/yr`** annualized upper-bound ceiling).


In [1]:
import numpy as np
import pandas as pd
from math import sqrt, erf

# Verified BigQuery Output from sql/02_funnel_and_markov.sql (Query 2.1 & 2.2)
funnel_df = pd.DataFrame([
    {"Cohort": "iOS (not_applicable)", "S0_Sessions": 128872, "S1_Plan_Select": 62532, "S2_Checkout": 43411, "S3_Converted": 16088,
     "S0_to_S1_Pct": 48.52, "S1_Drop_Hazard_Pct": 30.58, "S2_Drop_Hazard_Pct": 62.94, "Overall_CR_Pct": 12.48},
    {"Cohort": "Android Compatible (true)", "S0_Sessions": 85603, "S1_Plan_Select": 44666, "S2_Checkout": 25273, "S3_Converted": 8258,
     "S0_to_S1_Pct": 52.18, "S1_Drop_Hazard_Pct": 43.42, "S2_Drop_Hazard_Pct": 67.32, "Overall_CR_Pct": 9.65},
    {"Cohort": "Android Incompatible (false)", "S0_Sessions": 27073, "S1_Plan_Select": 7889, "S2_Checkout": 4735, "S3_Converted": 110,
     "S0_to_S1_Pct": 29.14, "S1_Drop_Hazard_Pct": 39.98, "S2_Drop_Hazard_Pct": 97.68, "Overall_CR_Pct": 0.41}
])
print(funnel_df.to_string(index=False))

# Exact Two-Proportion Z-Test (iOS vs. Android Compatible)
n1, x1 = 128872, 16088
n2, x2 = 85603, 8258
p1, p2 = x1 / n1, x2 / n2
p_pool = (x1 + x2) / (n1 + n2)
se_pool = sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z_stat = (p1 - p2) / se_pool
se_unpooled = sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
ci_low, ci_high = (p1 - p2) - 1.96*se_unpooled, (p1 - p2) + 1.96*se_unpooled
print(f"\nTwo-Proportion Z-Test: Diff = {(p1-p2)*100:.2f} pp (95% CI: [{ci_low*100:.2f} pp, {ci_high*100:.2f} pp]), Z = {z_stat:.2f}, p < 0.0001")


Cohort  S0_Sessions  S1_Plan_Select  S2_Checkout  S3_Converted  S0_to_S1_Pct  S1_Drop_Hazard_Pct  S2_Drop_Hazard_Pct  Overall_CR_Pct
        iOS (not_applicable)       128872           62532        43411         16088         48.52               30.58               62.94           12.48
   Android Compatible (true)        85603           44666        25273          8258         52.18               43.42               67.32            9.65
Android Incompatible (false)        27073            7889         4735           110         29.14               39.98               97.68            0.41

Two-Proportion Z-Test: Diff = 2.84 pp (95% CI: [2.57 pp, 3.10 pp]), Z = 20.28, p < 0.0001


In [2]:
# Verified BigQuery Output from sql/02_funnel_and_markov.sql (Query 2.3 & 2.4): Checkout Retries & Opportunity Sizing
retry_df = pd.DataFrame([
    {"Cohort": "iOS", "Attempt_1_CR": "37.47% (14,987/39,998)", "Attempt_2_CR": "35.15% (930/2,646)", "Attempt_3Plus_CR": "17.92% (132/736)", "Relative_Drop_1_to_3Plus": "-52.18%"},
    {"Cohort": "Android Compatible", "Attempt_1_CR": "32.22% (7,418/23,023)", "Attempt_2_CR": "37.99% (658/1,732)", "Attempt_3Plus_CR": "30.24% (150/496)", "Relative_Drop_1_to_3Plus": "-6.15%"},
    {"Cohort": "Android Incompatible", "Attempt_1_CR": "2.35% (98/4,178)", "Attempt_2_CR": "2.18% (9/413)", "Attempt_3Plus_CR": "1.95% (3/154)", "Relative_Drop_1_to_3Plus": "-17.02%"}
])
print("Checkout Retry Decay (click_to_order_2):")
print(retry_df.to_string(index=False))

# Upper-Bound Hardware Opportunity Sizing
incompat_sessions = 27073
compat_cr = 8258 / 85603
compat_arppu = 116945.46 / 8258
potential_buyers_rounded = round(incompat_sessions * compat_cr) # 2,612
gross_rev_rounded = potential_buyers_rounded * round(compat_arppu, 4)
net_forfeited_27d = gross_rev_rounded - 1238.19
annualized_ceiling = net_forfeited_27d * (365 / 27)
print(f"\nUpper-Bound Opportunity: {potential_buyers_rounded} potential buyers -> +${net_forfeited_27d:,.2f} net over 27d (~${annualized_ceiling:,.0f}/yr ceiling)")


Checkout Retry Decay (click_to_order_2):
              Cohort           Attempt_1_CR       Attempt_2_CR  Attempt_3Plus_CR Relative_Drop_1_to_3Plus
                 iOS 37.47% (14,987/39,998) 35.15% (930/2,646)  17.92% (132/736)                  -52.18%
  Android Compatible  32.22% (7,418/23,023) 37.99% (658/1,732)  30.24% (150/496)                   -6.15%
Android Incompatible       2.35% (98/4,178)     2.18% (9/413)     1.95% (3/154)                  -17.02%

Upper-Bound Opportunity: 2612 potential buyers -> +$35,751.65 net over 27d (~$483,310/yr ceiling)
